# ST-MEM으로 PTB-XL 사전학습·복원·분류 평가
기존 `Masking_pilot.ipynb`는 보존합니다. 이 노트북과 서버 runner는 `run_st_mem_pilot.py`를 공통 실행 코드로 사용합니다.

한 번에 한 조건을 실행하여 **ST-MEM 학습과 분류 효과를 확인**합니다. patch 50/125/250 × masking 50/75/90%의 9조건을 비교할 때는 README의 서버 제출 반복문을 사용합니다. 기본 pilot은 공식 Small(encoder 384차원·12층, decoder 256차원·4층), patch 50, nominal mask 75%, 1,000 update입니다.

**논문 설정 그대로의 재현은 아닙니다.** 기존 250 Hz·10초·lead별 z-score 데이터를 사용하고, raw 전처리 파형 복원을 평가하기 위해 `norm_pix_loss=False`로 둡니다. 원 논문의 crop/정규화/학습 recipe와 다릅니다. 공식 가중치는 자동으로 받지 않고 처음부터 학습합니다.

원본 코드: [VUNO ST-MEM](https://github.com/vuno/ST-MEM), Na et al., ICLR 2024. 별도 clone의 LICENSE(CC BY-NC 4.0)와 attribution을 유지하세요.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Image
cwd = Path.cwd().resolve()
EXPERIMENTS = next((p for root in [cwd, *cwd.parents]
    for p in [root, root / 'experiments', root / 'predictability-guided-ecg-masking' / 'experiments']
    if (p / 'run_st_mem_pilot.py').is_file()), None)
if EXPERIMENTS is None:
    raise FileNotFoundError('run_st_mem_pilot.py가 있는 experiments 폴더에서 실행하세요.')
if str(EXPERIMENTS) not in sys.path:
    sys.path.insert(0, str(EXPERIMENTS))
from run_st_mem_pilot import arguments, prepare_experiment, check_model, train_experiment


## 1. 실행 설정
로컬 CPU에서는 smoke만 실행합니다. smoke 기본 모델은 축소된 동작 검사 모델이므로 연구 결과가 아닙니다.
GPU에 할당된 뒤 MODE='pilot', RUN_TRAINING=True로 바꾸면 공식 Small로 학습합니다. A2000 12GB를 위해 microbatch 4 × accumulation 8 = effective batch 32를 기본으로 둡니다. 메모리는 GPU smoke로 확인해야 합니다.

서버 예: ST_MEM_REPO='/data/gykwak03/capstone2/ST-MEM', ARTIFACT_DIR='/data/gykwak03/capstone2/ptbxl_250hz'. 전처리는 다시 실행하지 않습니다.


In [ ]:
MODE = 'smoke'
RUN_TRAINING = False
TINY_OVERFIT = False  # RUN_TRAINING과 동시에 켜지 않습니다.
MODEL_SIZE = None     # None: smoke=축소 모델, pilot=공식 small. GPU smoke에서는 'small' 사용 가능.
ST_MEM_REPO = None    # None: 로컬 workspace의 ST-MEM clone
ARTIFACT_DIR = None   # None: 기존 preprocessing/artifacts/ptbxl_250hz
OUTPUT_ROOT = None
PATCH_SAMPLES = 50
MASK_RATIO = 0.75
SEED = 42
STEPS = None          # 기본 smoke=8, pilot=1000 optimizer updates
BATCH_SIZE = None     # 기본 smoke=2, pilot=4 microbatch
ACCUM_STEPS = None    # 기본 smoke=1, pilot=8
RECONSTRUCTION_ONLY = False  # True면 두 encoder의 분류 비교를 생략
argv = ['--mode', MODE, '--seed', str(SEED), '--patch-samples', str(PATCH_SAMPLES),
        '--mask-ratio', str(MASK_RATIO), '--subset-seed', '42', '--eval-mask-seed', '42']
for flag, value in [('--model-size', MODEL_SIZE), ('--st-mem-repo', ST_MEM_REPO),
    ('--artifact-dir', ARTIFACT_DIR), ('--output-root', OUTPUT_ROOT), ('--steps', STEPS),
    ('--batch-size', BATCH_SIZE), ('--accum-steps', ACCUM_STEPS)]:
    if value is not None:
        argv += [flag, str(value)]
if RUN_TRAINING: argv += ['--train']
if TINY_OVERFIT: argv += ['--tiny-overfit']
if RECONSTRUCTION_ONLY: argv += ['--no-probe']
options = arguments(argv)


## 2. 데이터와 공식 모델 연결
기존 노트북의 데이터 계약과 patient-disjoint train/validation 분할을 재사용합니다. train fold 1–8, validation fold 9이며 test는 열지 않습니다. pilot train 5,000 ECG subset·평가 mask seed는 42로 고정합니다.

ST-MEM은 각 lead에서 같은 개수의 visible patch를 남깁니다. visible 개수는 Decimal 기반 floor 계산으로 정수 경계의 소수점 오차를 제거합니다. patch 50/mask 75%는 lead당 50개 중 12개만 보여서 **실제 mask는 76%**입니다. mask 90%에서는 patch 50/125/250 모두 정확히 90%를 가리며, patch 250에서도 lead당 1개를 보여줍니다. 이전 binary-float 방식의 92%/95% 결과와 구분하고 nominal과 실제 비율을 따로 저장합니다. 기존 global random 75%와 같은 mask가 아니므로 architecture만의 공정한 교체 실험이라고 주장하지 않습니다.

`st_mem_adapter.py`는 외부 mask를 공식 masking 단계에 주입합니다. 위치 encoding, lead/SEP embedding, encoder, lead-wise decoder, feature pooling은 원본 함수를 사용합니다. 전처리를 중복 적용하지 않습니다.


In [ ]:
context = prepare_experiment(options)
config = context['config']
display({key: config[key] for key in ['model_size', 'parameters', 'device',
    'architecture', 'patch_samples', 'nominal_mask_ratio', 'realized_mask_ratio',
    'microbatch_size', 'accumulation_steps', 'effective_batch_size', 'norm_pix_loss']})
print('train ECG:', len(config['train_ecg_ids']), '| validation ECG:', len(config['validation_ecg_ids']))


## 3. 학습 전에 검사
Patchify/unpatchify가 lead·시간 순서를 보존하는지, hidden 입력값을 바꿔도 예측은 같은지, forward/backward에 NaN/Inf가 없는지 검사합니다. 검사에는 optimizer update가 없으므로 모델 가중치는 바뀌지 않습니다.


In [ ]:
check_model(context)


## 4. 사전학습과 평가
가린 patch에만 MSE를 적용합니다. AdamW와 warmup/cosine을 사용하고 accumulation이 끝난 뒤 한 번만 gradient clipping·optimizer update·scheduler update를 합니다.

고정 train/validation ECG·mask의 곡선을 200 update마다 기록합니다. 마지막에는 validation 최대 128 ECG × 3 mask를 평가하고 정답·예측·mask NPZ/PNG, 0 예측 대비 개선, RMS·상관계수, train-fit 선형 복원 기준을 저장합니다.

분류 비교는 같은 초기 가중치의 미학습 encoder와 학습된 encoder를 각각 고정하고, 동일한 정답 데이터·학습량으로 선형 분류층만 학습합니다. feature 표준화는 train에서만 fit합니다. ST-MEM의 공식 pooling을 사용하므로 이전 모델의 pooling과는 다릅니다.

`initial_checkpoint.pt`, 중간 `latest_checkpoint.pt`, 최종 `checkpoint.pt`를 저장합니다. optimizer/RNG 상태를 포함하는 resume checkpoint는 아닙니다. 매 실행은 새 폴더라 기존 결과는 덮어쓰지 않습니다.


In [ ]:
if RUN_TRAINING or TINY_OVERFIT:
    outcome = train_experiment(context)
    RUN_DIR = outcome['RUN_DIR']
    RESULTS = outcome['RESULTS']
    print('저장 경로:', RUN_DIR)
else:
    RUN_DIR = None
    print('검사만 완료했습니다. 학습하려면 설정 셀을 바꾸고 전체 셀을 다시 실행하세요.')


## 5. 결과 읽기
`validation_skill_vs_zero = 1 - model_MSE / zero_MSE`: 높을수록 0 예측보다 잘 복원합니다. RMS 비율은 출력 크기, correlation은 정답과의 모양 일치를 나타냅니다. visible input을 복사한 구간은 복원 점수에 포함하지 않습니다.

AUROC 절대값뿐 아니라 `auroc_gain_vs_random`을 보세요. 1-seed·단기 pilot 결과만으로 연구 방법의 효과를 확정하지 않습니다. 기본 모델이 작동하는지 확인한 뒤 같은 ST-MEM 안에서 masking policy를 비교하세요.


In [ ]:
if RUN_DIR is not None and not RESULTS.empty:
    keys = ['model_size', 'realized_mask_ratio', 'validation_masked_mse',
            'validation_skill_vs_zero', 'validation_rms_ratio', 'validation_masked_correlation',
            'macro_auroc', 'random_encoder_macro_auroc', 'auroc_gain_vs_random']
    display(RESULTS[[key for key in keys if key in RESULTS]])
if RUN_DIR is not None:
    condition = RUN_DIR / f'p{PATCH_SAMPLES}_r{int(round(MASK_RATIO*100)):02d}'
    for path in [condition / 'learning_curves.png', condition / 'diagnostic_curves.png',
                 *sorted(condition.glob('final_reconstruction_ecg*.png'))[:1]]:
        if path.is_file():
            display(Image(filename=str(path)))
